In [7]:
import os
import sys
import re
from pathlib import Path
from typing import List, Dict, Any

from PIL import Image
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import TrOCRProcessor, VisionEncoderDecoderModel

from PySide6 import QtCore, QtGui, QtWidgets


# ---------------------------------------------------------------------
# CONFIG / PATHS
# ---------------------------------------------------------------------
TEST_IMG_DIR = Path(r"D:\JLR_VIN_2549\ocr_test\images")
# TEST_LBL_DIR = Path(r"D:\JLR_VIN_2549\ocr_test\labels")
MODEL_PATH   = Path(r"D:\JLR_VIN_2549\best_trocr_v1\best_trocr_v1")

RESULTS_DIR   = TEST_IMG_DIR.parent / "eval_results_gui"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CONFUSION_PNG = RESULTS_DIR / "ocr_test_confusion_matrix_gui.png"


# ---------------------------------------------------------------------
# SAME CLEANING FUNCTION YOU USED FOR TRAINING / NON-GUI EVAL
# ---------------------------------------------------------------------
def clean_vin_text(text: str) -> str:
    cleaned = text.upper()
    replacements = {
        'O': '0',
        'I': '1',
        'Z': '2',
        'B': '8',
        '/': '',
        '.': '',
        ' ': '',
    }
    for wrong, correct in replacements.items():
        cleaned = cleaned.replace(wrong, correct)

    cleaned = re.sub(r'[^A-Z0-9*]', '', cleaned)
    middle = cleaned.strip('*')

    if len(middle) == 17:
        return f"*{middle}*"
    elif len(middle) > 17:
        return f"*{middle[:17]}*"
    else:
        middle = middle.ljust(17, 'X')
        return f"*{middle}*"


# ---------------------------------------------------------------------
# LOAD MODEL (GLOBAL, SHARED BY WORKER)
# ---------------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

processor = TrOCRProcessor.from_pretrained(str(MODEL_PATH))
model = VisionEncoderDecoderModel.from_pretrained(str(MODEL_PATH))
model.to(device)
model.eval()
print("Model loaded.")


# ---------------------------------------------------------------------
# WORKER THREAD: RUN EVAL ON TEST SET (NON-BLOCKING GUI)
# ---------------------------------------------------------------------
class EvalWorker(QtCore.QThread):
    """
    Runs the SAME loop as the non-GUI eval:
      - uses TEST_IMG_DIR,, clean_vin_text
      - predicts with TrOCR
      - sends each sample back to GUI
    """
    progress  = QtCore.Signal(int)
    item_done = QtCore.Signal(dict)
    finished  = QtCore.Signal()

    def __init__(self, parent: QtCore.QObject = None):
        super().__init__(parent)
        self._stop = False

    def stop(self):
        self._stop = True

    def run(self):
        test_images = sorted([p for p in TEST_IMG_DIR.glob("*.jpg")])
        print("-->",len(test_images))
        total = len(test_images)
        print(f"Worker: found {total} test images")
        for idx, img_path in enumerate(test_images, start=1):
            if self._stop:
                break

            # lbl_path = TEST_LBL_DIR / f"{img_path.stem}.txt"

            # if not lbl_path.exists():
            #     print(f"⚠ No label file found for: {img_path.name}")
            #     # still update progress
            #     self.progress.emit(int(idx * 100 / max(total, 1)))
            #     continue

            try:
                raw_lbl = str(img_path).split("\\")[-1].split("_")[0]
                print(" raw_lbl --->",raw_lbl)
            except Exception as e:
                print(f"Error reading label {raw_lbl}: {e}")
                self.progress.emit(int(idx * 100 / max(total, 1)))
                continue

            gt_clean = clean_vin_text(raw_lbl)
            
            print("gt_clean --->",gt_clean)

            try:
                img = Image.open(img_path).convert("RGB")
            except Exception as e:
                print(f"Error opening image {img_path}: {e}")
                self.progress.emit(int(idx * 100 / max(total, 1)))
                continue

            # Predict VIN (SAME STYLE AS NON-GUI SCRIPT)
            try:
                pixel_values = processor(images=img, return_tensors="pt").pixel_values.to(device)
                with torch.no_grad():
                    ids = model.generate(pixel_values, max_length=25)
                    pred_raw = processor.batch_decode(ids, skip_special_tokens=True)[0]
            except Exception as e:
                pred_raw = f"[ERROR: {e}]"

            # We also compute a cleaned version for display, but metrics
            # will use gt_clean + pred_raw (like the non-GUI code).
            if pred_raw.startswith("[ERROR"):
                pred_clean = ""
            else:
                pred_clean = clean_vin_text(pred_raw)

            row = {
                "path": str(img_path),
                "image_name": img_path.name,
                "gt_raw": raw_lbl,
                "gt_clean": gt_clean,
                "pred_raw": pred_raw,
                "pred_clean": pred_clean,
            }

            self.item_done.emit(row)
            self.progress.emit(int(idx * 100 / max(total, 1)))

        self.finished.emit()


# ---------------------------------------------------------------------
# MAIN WINDOW (OLD STRUCTURE: PREVIEW + TABLE + CONTROLS)
# ---------------------------------------------------------------------
class MainWindow(QtWidgets.QMainWindow):
    def __init__(self):
        super().__init__()

        self.setWindowTitle("VIN OCR Evaluation – TrOCR (Test Set GUI)")
        self.resize(1400, 800)

        self.rows: List[Dict[str, Any]] = []
        self.worker: EvalWorker | None = None

        self._setup_ui()
        self._connect_signals()

    # ---------------- UI LAYOUT (OLD STRUCTURE) ----------------
    def _setup_ui(self):
        # Left: image preview
        self.imageLabel = QtWidgets.QLabel("Preview")
        self.imageLabel.setAlignment(QtCore.Qt.AlignCenter)
        self.imageLabel.setMinimumSize(800, 600)
        self.imageLabel.setStyleSheet(
            """
            QLabel {
                background: #111;
                color: #999;
                border: 2px solid #444;
                font-size: 12px;
            }
            """
        )

        # Controls group (top-left under preview)
        control_group = QtWidgets.QGroupBox("Controls")
        control_layout = QtWidgets.QGridLayout(control_group)

        self.btnRunEval  = QtWidgets.QPushButton("▶ Run Eval (Test Set)")
        self.btnClear    = QtWidgets.QPushButton("🗑 Clear")
        self.btnMetrics  = QtWidgets.QPushButton("📊 Show Metrics")
        self.btnExit     = QtWidgets.QPushButton("❌ Exit")

        self.lblInfo = QtWidgets.QLabel(
            f"Images: {TEST_IMG_DIR}"
        )

        self.progress = QtWidgets.QProgressBar()
        self.progress.setRange(0, 100)
        self.progress.setValue(0)

        control_layout.addWidget(self.lblInfo,   0, 0, 1, 4)
        control_layout.addWidget(self.btnRunEval, 1, 0)
        control_layout.addWidget(self.btnMetrics, 1, 1)
        control_layout.addWidget(self.btnClear,   1, 2)
        control_layout.addWidget(self.btnExit,    1, 3)
        control_layout.addWidget(self.progress,   2, 0, 1, 4)

        # Right: table with results
        self.table = QtWidgets.QTableWidget(0, 6)
        self.table.setHorizontalHeaderLabels(
            ["Image", "GT (clean)", "Pred (clean)", "Status", "CharAcc%", "Details"]
        )
        self.table.horizontalHeader().setStretchLastSection(True)
        self.table.setSelectionBehavior(QtWidgets.QAbstractItemView.SelectRows)
        self.table.setEditTriggers(QtWidgets.QAbstractItemView.NoEditTriggers)
        self.table.setColumnWidth(0, 250)
        self.table.setColumnWidth(1, 200)
        self.table.setColumnWidth(2, 200)
        self.table.setColumnWidth(3, 100)
        self.table.setColumnWidth(4, 80)
        self.table.setColumnWidth(5, 300)

        # Layout the main window (old structure: left preview + controls, right table)
        root_widget = QtWidgets.QWidget()
        main_layout = QtWidgets.QHBoxLayout(root_widget)

        left_layout = QtWidgets.QVBoxLayout()
        left_layout.addWidget(self.imageLabel, 3)
        left_layout.addWidget(control_group, 1)

        right_layout = QtWidgets.QVBoxLayout()
        right_layout.addWidget(QtWidgets.QLabel("<b>Results:</b>"))
        right_layout.addWidget(self.table, 1)

        main_layout.addLayout(left_layout, 2)
        main_layout.addLayout(right_layout, 1)

        self.setCentralWidget(root_widget)

    def _connect_signals(self):
        self.btnRunEval.clicked.connect(self.run_eval)
        self.btnClear.clicked.connect(self.clear_results)
        self.btnMetrics.clicked.connect(self.show_metrics)
        self.btnExit.clicked.connect(self.exit_application)
        self.table.itemSelectionChanged.connect(self.on_row_selected)

    # ---------------- Worker lifecycle ----------------
    def _stop_worker(self):
        if self.worker and self.worker.isRunning():
            self.worker.stop()
            self.worker.wait(2000)

    def closeEvent(self, event: QtGui.QCloseEvent):
        self._stop_worker()
        event.accept()

    def exit_application(self):
        self._stop_worker()
        self.close()

    # ---------------- Actions ----------------
    def clear_results(self):
        self._stop_worker()
        self.rows.clear()
        self.table.setRowCount(0)
        self.imageLabel.clear()
        self.imageLabel.setText("Preview")
        self.progress.setValue(0)

    def run_eval(self):
        self.clear_results()

        # Start worker over TEST_IMG_DIR
        self.worker = EvalWorker(self)
        self.worker.item_done.connect(self.add_result_row)
        self.worker.progress.connect(self.progress.setValue)
        self.worker.finished.connect(self.on_eval_finished)
        self.worker.start()

    @QtCore.Slot(dict)
    def add_result_row(self, result: dict):
        """
        Called for each image from worker.
        result keys:
          - path, image_name, gt_raw, gt_clean, pred_raw, pred_clean
        """
        self.rows.append(result)

        row_idx = self.table.rowCount()
        self.table.insertRow(row_idx)

        # Display image name
        img_item = QtWidgets.QTableWidgetItem(result["image_name"])
        img_item.setToolTip(result["path"])
        self.table.setItem(row_idx, 0, img_item)

        # GT clean
        self.table.setItem(row_idx, 1, QtWidgets.QTableWidgetItem(result["gt_clean"]))

        # Pred clean
        self.table.setItem(row_idx, 2, QtWidgets.QTableWidgetItem(result["pred_clean"]))

        # Status: EXACT or MISMATCH, using SAME logic as non-GUI:
        # exact_matches = sum(gt_clean == pred_raw)
        if result["pred_raw"].startswith("[ERROR"):
            status_str = "ERROR"
            status_color = QtGui.QColor(128, 128, 128)
            bg_color = QtGui.QColor(240, 240, 240)
        else:
            if result["gt_clean"] == result["pred_raw"]:
                status_str = "EXACT"
                status_color = QtGui.QColor(0, 180, 0)
                bg_color = QtGui.QColor(240, 255, 240)
            else:
                status_str = "MISMATCH"
                status_color = QtGui.QColor(220, 0, 0)
                bg_color = QtGui.QColor(255, 240, 240)

        status_item = QtWidgets.QTableWidgetItem(status_str)
        status_item.setForeground(QtGui.QBrush(status_color))
        status_item.setBackground(QtGui.QBrush(bg_color))
        self.table.setItem(row_idx, 3, status_item)

        # Char-level accuracy per sample (similar style to global metric, but per row)
        char_acc = self._per_sample_char_accuracy(result["gt_clean"], result["pred_raw"])
        acc_item = QtWidgets.QTableWidgetItem(f"{char_acc:.1f}")
        self.table.setItem(row_idx, 4, acc_item)

        # Details: show raw GT and raw pred
        details = f"GT raw: {result['gt_raw']} | Pred raw: {result['pred_raw']}"
        self.table.setItem(row_idx, 5, QtWidgets.QTableWidgetItem(details))

        # For the first row, show image in preview
        if row_idx == 0:
            self._show_image(result["path"])

    def _per_sample_char_accuracy(self, gt_clean: str, pred_raw: str) -> float:
        """
        Per-sample char accuracy with the same style:
          - use gt_clean and pred_raw
          - pad with '*' to same length
        """
        if not gt_clean or not pred_raw or pred_raw.startswith("[ERROR"):
            return 0.0

        L = max(len(gt_clean), len(pred_raw))
        gt = gt_clean.ljust(L, "*")
        pr = pred_raw.ljust(L, "*")

        total = len(gt)
        correct = sum(1 for g, p in zip(gt, pr) if g == p)
        return (correct / total * 100.0) if total > 0 else 0.0

    def _show_image(self, path: str):
        pixmap = QtGui.QPixmap(path)
        if not pixmap.isNull():
            scaled = pixmap.scaled(
                self.imageLabel.size(),
                QtCore.Qt.KeepAspectRatio,
                QtCore.Qt.SmoothTransformation,
            )
            self.imageLabel.setPixmap(scaled)
        else:
            self.imageLabel.setText("Failed to load image.")

    def on_row_selected(self):
        selected = self.table.selectionModel().selectedRows()
        if not selected:
            return
        row_idx = selected[0].row()
        if row_idx < 0 or row_idx >= len(self.rows):
            return
        path = self.rows[row_idx]["path"]
        self._show_image(path)

    def on_eval_finished(self):
        QtWidgets.QMessageBox.information(
            self,
            "Evaluation finished",
            f"Processed {len(self.rows)} labeled images from:\n{TEST_IMG_DIR}",
        )

    # ---------------- Metrics (same logic as non-GUI script) ----------------
    def show_metrics(self):
        if not self.rows:
            QtWidgets.QMessageBox.warning(
                self,
                "No data",
                "No evaluation results available. Run the evaluation first.",
            )
            return

        # Build gt_texts and pred_texts EXACTLY as the non-GUI script:
        gt_texts   = [r["gt_clean"] for r in self.rows]
        pred_texts = [r["pred_raw"]  for r in self.rows]

        # Exact match accuracy
        exact_matches = sum(1 for gt, pred in zip(gt_texts, pred_texts) if gt == pred)
        exact_accuracy = exact_matches / len(gt_texts) if gt_texts else 0.0

        # Character-level confusion matrix (same logic)
        chars = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789*")
        char_to_idx = {c: i for i, c in enumerate(chars)}
        matrix = np.zeros((len(chars), len(chars)), dtype=int)

        total_chars = 0
        correct_chars = 0

        for gt, pred in zip(gt_texts, pred_texts):
            L = max(len(gt), len(pred))
            gt_pad = gt.ljust(L, "*")
            pred_pad = pred.ljust(L, "*")

            for g, p in zip(gt_pad, pred_pad):
                if g in char_to_idx and p in char_to_idx:
                    matrix[char_to_idx[g], char_to_idx[p]] += 1

                if g == p:
                    correct_chars += 1

                total_chars += 1

        char_accuracy = correct_chars / total_chars if total_chars > 0 else 0.0

        # Show summary in a message box
        msg = (
            f"Samples (with labels): {len(gt_texts)}\n"
            f"Exact matches: {exact_matches}\n"
            f"Exact match accuracy: {exact_accuracy:.4f}\n\n"
            f"Total characters: {total_chars}\n"
            f"Character-level accuracy: {char_accuracy:.4f}\n"
        )
        QtWidgets.QMessageBox.information(self, "Evaluation Metrics", msg)

        # Plot and show confusion matrix (same style as non-GUI code)
        plt.figure(figsize=(18, 18))
        sns.heatmap(
            matrix,
            xticklabels=chars,
            yticklabels=chars,
            cmap="Blues",
            fmt="d"
        )
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.title("VIN Character-Level Confusion Matrix")
        plt.tight_layout()
        plt.savefig(CONFUSION_PNG)
        plt.show()

        print("Confusion matrix saved to:", CONFUSION_PNG)


# ---------------------------------------------------------------------
# ENTRY POINT
# ---------------------------------------------------------------------
def main():
    app = QtWidgets.QApplication.instance()
    if not app:
        app = QtWidgets.QApplication(sys.argv)

    app.setStyle("Fusion")
    # Simple dark palette (like your old GUI style)
    dark = QtGui.QPalette()
    dark.setColor(QtGui.QPalette.Window, QtGui.QColor(53, 53, 53))
    dark.setColor(QtGui.QPalette.WindowText, QtGui.QColor(255, 255, 255))
    dark.setColor(QtGui.QPalette.Base, QtGui.QColor(25, 25, 25))
    dark.setColor(QtGui.QPalette.AlternateBase, QtGui.QColor(53, 53, 53))
    dark.setColor(QtGui.QPalette.ToolTipBase, QtGui.QColor(255, 255, 255))
    dark.setColor(QtGui.QPalette.ToolTipText, QtGui.QColor(255, 255, 255))
    dark.setColor(QtGui.QPalette.Text, QtGui.QColor(255, 255, 255))
    dark.setColor(QtGui.QPalette.Button, QtGui.QColor(53, 53, 53))
    dark.setColor(QtGui.QPalette.ButtonText, QtGui.QColor(255, 255, 255))
    dark.setColor(QtGui.QPalette.BrightText, QtGui.QColor(255, 0, 0))
    dark.setColor(QtGui.QPalette.Highlight, QtGui.QColor(42, 130, 218))
    dark.setColor(QtGui.QPalette.HighlightedText, QtGui.QColor(0, 0, 0))
    app.setPalette(dark)

    window = MainWindow()
    window.show()

    sys.exit(app.exec())


if __name__ == "__main__":
    main()


Using device: cpu


Config of the encoder: <class 'transformers.models.vit.modeling_vit.ViTModel'> is overwritten by shared encoder config: ViTConfig {
  "attention_probs_dropout_prob": 0.0,
  "encoder_stride": 16,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.0,
  "hidden_size": 768,
  "image_size": 384,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "model_type": "vit",
  "num_attention_heads": 12,
  "num_channels": 3,
  "num_hidden_layers": 12,
  "patch_size": 16,
  "qkv_bias": false,
  "transformers_version": "4.48.0"
}

Config of the decoder: <class 'transformers.models.trocr.modeling_trocr.TrOCRForCausalLM'> is overwritten by shared decoder config: TrOCRConfig {
  "activation_dropout": 0.0,
  "activation_function": "gelu",
  "add_cross_attention": true,
  "attention_dropout": 0.0,
  "bos_token_id": 0,
  "classifier_dropout": 0.0,
  "cross_attention_hidden_size": 768,
  "d_model": 1024,
  "decoder_attention_heads": 16,
  "decoder_ffn_dim": 4096,
  "decoder

Model loaded.
--> 160
Worker: found 160 test images
 raw_lbl ---> SAL119E95SA606669
gt_clean ---> *SAL119E95SA606669*
 raw_lbl ---> SAL1A2A43SA606882
gt_clean ---> *SAL1A2A43SA606882*
 raw_lbl ---> SAL1A2A47SA606660
gt_clean ---> *SAL1A2A47SA606660*
 raw_lbl ---> SAL1A2A48SA605985
gt_clean ---> *SAL1A2A48SA605985*
 raw_lbl ---> SAL1A2A49SA606627
gt_clean ---> *SAL1A2A49SA606627*
 raw_lbl ---> SAL1A2A92SA606173
gt_clean ---> *SAL1A2A92SA606173*
 raw_lbl ---> SAL1A2AW0SA605416
gt_clean ---> *SAL1A2AW0SA605416*
 raw_lbl ---> SAL1A2AW0SA606839
gt_clean ---> *SAL1A2AW0SA606839*
 raw_lbl ---> SAL1A2AW1SA606607
gt_clean ---> *SAL1A2AW1SA606607*
 raw_lbl ---> SAL1A2AW2SA606647
gt_clean ---> *SAL1A2AW2SA606647*
 raw_lbl ---> SAL1A2AW2SA606812
gt_clean ---> *SAL1A2AW2SA606812*
 raw_lbl ---> SAL1A2AW2SA606969
gt_clean ---> *SAL1A2AW2SA606969*
 raw_lbl ---> SAL1A2AW4SA606844
gt_clean ---> *SAL1A2AW4SA606844*
 raw_lbl ---> SAL1A2AW4SA606875
gt_clean ---> *SAL1A2AW4SA606875*
 raw_lbl ---> SAL1A2AW5S

SystemExit: 0